In [27]:
import polars as pl
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt
import altair as alt

import itables
import itables.options as opt

import statsmodels.api as sm

In [2]:
alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

In [3]:
itables.init_notebook_mode(all_interactive=True)
opt.maxBytes = 0          # no byte limit (default is ~64KB)
opt.maxRows = 0           # no row limit
opt.maxColumns = 0        # no column limit

In [4]:
df = pl.read_parquet("1d_candle_panel.parquet")

# Filter leading zero volume candles

In [5]:
df = df.sort(["symbol", "timestamp"]).filter(
        (pl.col("volume") != 0)
            .cum_max()
            .over("symbol")
    ).sort(["symbol", "timestamp"])

# Compute 30 day median daily dollar volume

In [6]:
df = df.with_columns(
    (pl.col("volume") * pl.col("close")).over("symbol").alias("dollar_volume")
)

In [7]:
df = df.with_columns(
    pl.col("dollar_volume").rolling_median(30).over("symbol").alias("30_mdv")
)

In [8]:
# Lag the signal by one day so the 30 day median dollar volume through t-1
# ranks the return earned at t (without the lag the signal uses day t's own volume)

df = df.with_columns(
    pl.col("30_mdv").shift(1).over("symbol").alias("30_mdv_lag")
)

In [9]:
df = df.drop_nans().drop_nulls()

# Label symbols by 30 day median daily volume quantile by date

In [10]:
df = df.with_columns(
    pl.col("30_mdv_lag")
    .bin_quantiles([0.3, 0.7], labels=["low", "mid", "high"])
    .over("timestamp")
    .alias("quantile")
)

In [11]:
df.group_by("symbol").agg(pl.col("30_mdv").mean())

shape: (234, 2)
┌────────┬───────────────┐
│ symbol ┆ 30_mdv        │
│ ---    ┆ ---           │
│ str    ┆ f64           │
╞════════╪═══════════════╡
│ 0G     ┆ 493423.860512 │
│ 2Z     ┆ 357650.510848 │
│ AAVE   ┆ 1.0441e7      │
│ ACE    ┆ 367577.521641 │
│ ADA    ┆ 6.2075e6      │
│ …      ┆ …             │
│ kFLOKI ┆ 1.3096e6      │
│ kLUNC  ┆ 326754.572014 │
│ kNEIRO ┆ 718898.422216 │
│ kPEPE  ┆ 2.8004e7      │
│ kSHIB  ┆ 2.3461e6      │
└────────┴───────────────┘

# Create long panel dataframe of geometric equal weighted indexes for each liquidity quantile

In [12]:
indexes = df.group_by(["timestamp", "quantile"]).agg(
    pl.col("log_return").mean()
).sort(["quantile", "timestamp"])

In [13]:
indexes.group_by("quantile").agg(pl.col("timestamp").min())

shape: (3, 2)
┌──────────┬─────────────────────────┐
│ quantile ┆ timestamp               │
│ ---      ┆ ---                     │
│ enum     ┆ datetime[μs, UTC]       │
╞══════════╪═════════════════════════╡
│ low      ┆ 2023-04-04 00:00:00 UTC │
│ mid      ┆ 2023-03-29 00:00:00 UTC │
│ high     ┆ 2023-03-29 00:00:00 UTC │
└──────────┴─────────────────────────┘

In [14]:
# Make all quintiles have the same starting timestamp

indexes = indexes.filter(
    pl.col("timestamp") >= pl.col("timestamp").min().over("quantile").max()
)

In [15]:
indexes = indexes.with_columns(
    pl.col("log_return").cum_sum().exp().over("quantile").alias("close")
)

In [16]:
indexes

shape: (3_942, 4)
┌─────────────────────────┬──────────┬────────────┬──────────┐
│ timestamp               ┆ quantile ┆ log_return ┆ close    │
│ ---                     ┆ ---      ┆ ---        ┆ ---      │
│ datetime[μs, UTC]       ┆ enum     ┆ f64        ┆ f64      │
╞═════════════════════════╪══════════╪════════════╪══════════╡
│ 2023-04-04 00:00:00 UTC ┆ low      ┆ -0.013226  ┆ 0.986861 │
│ 2023-04-05 00:00:00 UTC ┆ low      ┆ 0.010139   ┆ 0.996918 │
│ 2023-04-06 00:00:00 UTC ┆ low      ┆ 0.005151   ┆ 1.002066 │
│ 2023-04-07 00:00:00 UTC ┆ low      ┆ -0.005477  ┆ 0.996593 │
│ 2023-04-08 00:00:00 UTC ┆ low      ┆ -0.005161  ┆ 0.991462 │
│ …                       ┆ …        ┆ …          ┆ …        │
│ 2026-11-03 00:00:00 UTC ┆ high     ┆ -0.04183   ┆ 0.141051 │
│ 2026-11-04 00:00:00 UTC ┆ high     ┆ 0.015049   ┆ 0.14319  │
│ 2026-11-05 00:00:00 UTC ┆ high     ┆ 0.023112   ┆ 0.146538 │
│ 2026-11-06 00:00:00 UTC ┆ high     ┆ -0.077873  ┆ 0.13556  │
│ 2026-11-07 00:00:00 UTC ┆ high     ┆ 0.019451   ┆ 0.138222 │
└─────────────────────────┴──────────┴────────────┴──────────┘

# Plot time series of all indexes

In [17]:
chart = (
    alt.Chart(indexes.sort("timestamp"))
    .mark_line(strokeWidth=1)
    .encode(
        x="timestamp:T",
        y=alt.Y("close:Q", scale=alt.Scale(zero=False)),
        color=alt.Color("quantile:N", sort=["low", "mid", "high"]),
        tooltip=["timestamp:T", "quantile:N", "close:Q"],
    )
    .properties(width=900, height=400)
    .interactive()
)
chart

alt.Chart(...)

# Compute volatility for all indexes

In [18]:
indexes = indexes.sort(["quantile", "timestamp"]).with_columns(
    pl.col("log_return").pow(2).ewm_mean(span=36, adjust=False, min_samples=36).sqrt().shift(1).over("quantile")
    .alias("vol")
).sort(["quantile", "timestamp"]).drop_nans().drop_nulls()

# Compute vol normalised log_returns for all indexes (annual target vol = 20%)

In [19]:
indexes = indexes.with_columns(
    (pl.col("log_return") * ((0.20 / np.sqrt(365)) / pl.col("vol")))
        .over("quantile").alias("vn_log_return")
)

In [20]:
high = indexes.filter(pl.col("quantile") == "high")
mid = indexes.filter(pl.col("quantile") == "mid")
low = indexes.filter(pl.col("quantile") == "low")

In [21]:
out = pl.DataFrame(
    [
        high["timestamp"],
        high["vn_log_return"].alias("high_vn_log_return"),
        low["vn_log_return"].alias("low_vn_log_return")
    ]
)

In [25]:
out = out.with_columns(
    pl.col("high_vn_log_return").shift(1).alias("high_vn_log_return_lag_1"),
    pl.col("low_vn_log_return").shift(1).alias("low_vn_log_return_lag_1")
).drop_nans().drop_nulls()

In [26]:
out

shape: (1_277, 5)
┌───────────────┬────────────────────┬────────────────────┬────────────────────┬───────────────────┐
│ timestamp     ┆ high_vn_log_return ┆ low_vn_log_return  ┆ high_vn_log_return ┆ low_vn_log_return │
│ ---           ┆ ---                ┆ ---                ┆ _lag_1             ┆ _lag_1            │
│ datetime[μs,  ┆ f64                ┆ f64                ┆ ---                ┆ ---               │
│ UTC]          ┆                    ┆                    ┆ f64                ┆ f64               │
╞═══════════════╪════════════════════╪════════════════════╪════════════════════╪═══════════════════╡
│ 2023-05-11    ┆ 0.007822           ┆ 0.000404           ┆ -0.000159          ┆ -0.002014         │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ 2023-05-12    ┆ -0.019175          ┆ -0.014611          ┆ 0.007822           ┆ 0.000404          │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ 2023-05-13    ┆ 0.013661           ┆ 0.006678           ┆ -0.019175          ┆ -0.014611         │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ 2023-05-14    ┆ -0.001609          ┆ -0.002988          ┆ 0.013661           ┆ 0.006678          │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ 2023-05-15    ┆ 0.003238           ┆ 0.003892           ┆ -0.001609          ┆ -0.002988         │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ …             ┆ …                  ┆ …                  ┆ …                  ┆ …                 │
│ 2026-11-03    ┆ -0.012192          ┆ -0.011097          ┆ -0.000609          ┆ -0.002108         │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ 2026-11-04    ┆ 0.004344           ┆ 0.003766           ┆ -0.012192          ┆ -0.011097         │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ 2026-11-05    ┆ 0.006827           ┆ 0.004702           ┆ 0.004344           ┆ 0.003766          │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ 2026-11-06    ┆ -0.023368          ┆ -0.027478          ┆ 0.006827           ┆ 0.004702          │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
│ 2026-11-07    ┆ 0.005295           ┆ 0.008541           ┆ -0.023368          ┆ -0.027478         │
│ 00:00:00 UTC  ┆                    ┆                    ┆                    ┆                   │
└───────────────┴────────────────────┴────────────────────┴────────────────────┴───────────────────┘